# 在 Jupyter 里跑通 ChatOpenAI（DeepSeek 后端）

**四个步骤，按顺序执行即可。**

1. 装依赖（`langchain-openai` + `python-dotenv`）
2. 拿到 DeepSeek API Key（https://platform.deepseek.com → API keys → 创建）
3. 把 Key 写进同目录的 `.env` 文件
4. 读取并调用

> 关键点：`.env` 必须和 **Jupyter 的工作目录** 在一起，而不是和 `.ipynb` 文件在一起。
> 第 3 个单元格会打印出工作目录，照那个路径放。


## 第 1 步 · 安装依赖

`langchain-openai` 需要 ≥ 0.2 才有 `usage_metadata` 字段。


In [2]:
%pip install -q --upgrade langchain-openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: D:\pythonproject\newpro\.venv\Scripts\python.exe -m pip install --upgrade pip


## 第 2 步 & 第 3 步 · 准备 `.env`

下面这个单元格：如果 `.env` 不存在就生成一个模板；存在就跳过。
生成后请打开它，把 `sk-xxxx` 换成你自己的 Key。


In [ ]:
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text(
        "DEEPSEEK_API_KEY=sk-在这里粘贴你的key\n"
        "DEEPSEEK_BASE_URL=https://api.deepseek.com\n",
        encoding="utf-8",
    )
    print("已生成 .env ->", env_path.resolve())
    print("请打开它填入你的 Key，然后【重启内核】再往下跑。")
else:
    print(".env 已存在 ->", env_path.resolve())


## 第 4 步 · 读取环境变量

两种写法任选其一。**改了 `.env` 必须重启内核**，因为 `load_dotenv()` 只在 import 那一刻读一次。


In [1]:
# 写法 A（纯 Python，推荐，跨环境最稳）
import os
from dotenv import load_dotenv

load_dotenv(override=True)          # override=True：让 .env 覆盖已有的同名环境变量

print("Jupyter 工作目录 =", os.getcwd())
print("%pwd 可用来看同一个东西")

KEY  = os.getenv("DEEPSEEK_API_KEY")
BASE = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

if not KEY or KEY.startswith("sk-在这里"):
    print("❌ 没读到 Key：请确认 .env 在工作目录里，且填好内容后重启了内核")
else:
    print("✅ 读到 Key：", KEY[:6] + "..." + KEY[-4:])
    print("✅ base_url：", BASE)


ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# 写法 B（IPython 魔法命令，等价，少写 import）
# %load_ext dotenv
# %dotenv


### 备用方案：不想落盘就用 `getpass` 临时输入

输入时不回显，也不会写进任何文件。每次重启内核要重输一次。


In [ ]:
import os
from getpass import getpass

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass("粘贴 DeepSeek API Key（输入时看不见）: ")
print("Key 已写入当前内核的环境变量")


## 跑你的原始代码（已修正模型名）

`deepseek-chat` / `deepseek-reasoner` 是 2026-07-24 起弃用的旧名，现在用 `deepseek-flash`（快、便宜）或 `deepseek-v4-pro`（强）。
旧名目前仍能调通，但会被路由到新模型，建议直接改掉。


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-flash",              # 换模型只改这一行；如需最强用 "deepseek-v4-pro"
    api_key=KEY,                          # 也可以写 api_key=os.environ["DEEPSEEK_API_KEY"]
    base_url=BASE,                        # 也可以写 base_url="https://api.deepseek.com"
)

msg = llm.invoke("用一句话解释什么是链式调用")

print(type(msg).__name__)   # AIMessage，不是 str
print(msg.content)          # 正文
print(msg.usage_metadata)   # 这次花了多少 token


### 如果 `usage_metadata` 打印出 `None`

说明 langchain-core 版本偏旧，或后端没回传该字段。退回读原始 usage 即可。


In [ ]:
if msg.usage_metadata is None:
    print(msg.response_metadata)


## 常见报错对照表

| 现象 | 原因 | 处理 |
| --- | --- | --- |
| `key` 是 `None` | `.env` 不在工作目录 / 改了没重启内核 | 打印 `os.getcwd()`，把 `.env` 放那儿；重启内核 |
| `AuthenticationError 401` | Key 填错、有多余空格或引号 | `.env` 里**不要**加引号：`DEEPSEEK_API_KEY=sk-xxx` |
| `NotFoundError 404 / model not found` | 模型名写错或用已下线的旧名 | 改成 `deepseek-flash` 或 `deepseek-v4-pro` |
| `Insufficient Balance 402` | 账户没余额 | 去 platform.deepseek.com 充值 |
| 改了 `.env` 没生效 | `load_dotenv()` 已执行过 | 菜单里重启内核，而不是只重跑单元格 |

**安全提醒**：把 `.env` 加进 `.gitignore`，永远不要把 Key 写进单元格再提交 notebook。
